# 05 — NeuralFP Pipeline (pfann)

Baut die pfann-Referenz-DB auf und führt alle Queries aus.

**Für den Live Run** wird der Score-Threshold **datengetrieben** auf dem
Dry-Run-Datensatz bestimmt (disjunkt zu den Live-Run-Songs):
- Dry-Run-Ref-DB (52 Songs) wird mit dem neuen Live-Modell indexiert
- Optimaler Threshold = F1-Maximum auf der Precision-Recall-Kurve

**Abhängigkeiten (Dry Run):**
- NB 00: `data/partitions/dry_ref.json`
- NB 01: `data/queries/manifest_dry.csv`
- NB 04: `checkpoints/dry_run/pfann_model/model.pt`, `pfann/configs/dry_run.json`

**Abhängigkeiten (Live Run):**
- NB 00: `data/partitions/dry_ref.json`, `live_ref.json`
- NB 01: `data/queries/manifest_dry.csv`, `manifest_live.csv`
- NB 04: `checkpoints/live_run/pfann_model/model.pt`, `pfann/configs/live_run.json`

**Ausgabe:** `results/{run_mode}_run/neuralFP_raw.csv`

In [ ]:
# ── RUN MODE CONFIG ──────────────────────────────────────────────────────────
# Switch between dry run and live run by editing src/run_config.py.
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve() / "src"))
from run_config import cfg
RUN_MODE = cfg.run_mode
print(f"RUN_MODE = {RUN_MODE!r}")

## 1. Imports und Pfade

In [ ]:
import os, sys, json, subprocess, time
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd

from run_config import cfg
RUN_MODE = cfg.run_mode

SEED = 42
np.random.seed(SEED)

PFANN_DIR     = PROJECT_ROOT / "src" / "pfann"
PFANN_LISTS   = PFANN_DIR / "lists"
PFANN_CONFIGS = PFANN_DIR / "configs"

# ── Mode-dependent paths ──────────────────────────────────────────────────────
CHECKPOINTS    = PROJECT_ROOT / "checkpoints" / cfg.checkpoints_subdir
RESULTS_DIR    = PROJECT_ROOT / "results" / cfg.results_subdir
MANIFEST_PATH  = PROJECT_ROOT / "data" / "queries" / cfg.manifest_name
CONFIG_PATH    = PFANN_CONFIGS / f"{cfg.pfann_config_name}.json"
LIST_PREFIX    = cfg.pfann_list_prefix       # "dry" or "live"
REF_LIST       = PFANN_LISTS / f"{LIST_PREFIX}_ref.txt"
QUERY_LIST     = PFANN_LISTS / f"{LIST_PREFIX}_queries.txt"
DB_DIR         = RESULTS_DIR / "pfann_db"
TSV_PATH       = RESULTS_DIR / "neuralFP_matches.tsv"
DETAIL_PATH    = RESULTS_DIR / "neuralFP_matches_detail.csv"
RAW_CSV_PATH   = RESULTS_DIR / "neuralFP_raw.csv"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN_MODE      = {RUN_MODE!r}")
print(f"MANIFEST_PATH = {MANIFEST_PATH}")
print(f"CONFIG_PATH   = {CONFIG_PATH}")
print(f"DB_DIR        = {DB_DIR}")
print("PROJECT_ROOT:", PROJECT_ROOT)

## 2. Voraussetzungen prüfen

Prüft ob NB 04 erfolgreich war (model.pt, dry_run.json, dry_ref.txt)
und ob NB 01 das Manifest erzeugt hat.

In [ ]:
assert CONFIG_PATH.exists(), (
    f"{CONFIG_PATH.name} fehlt: {CONFIG_PATH}\n"
    "NB 04 ausführen!"
)
with open(CONFIG_PATH) as fh:
    run_pfann_config = json.load(fh)

model_dir = Path(run_pfann_config["model_dir"])
model_pt  = model_dir / "model.pt"

required = {
    f"manifest_{RUN_MODE}.csv":         MANIFEST_PATH,
    f"{LIST_PREFIX}_ref.txt":           REF_LIST,
    f"{cfg.pfann_config_name}.json":    CONFIG_PATH,
    "model.pt (aus NB 04)":             model_pt,
}
all_ok = True
for label, p in required.items():
    status = "OK" if p.exists() else "FEHLT"
    print(f"  [{status}] {label}: {p}")
    if not p.exists():
        all_ok = False

if not all_ok:
    raise FileNotFoundError(
        "Nicht alle Voraussetzungen erfüllt — "
        "NB 00, NB 01 und NB 04 müssen zuerst ausgeführt werden."
    )
print("\nAlle Voraussetzungen erfüllt. ✓")
print(f"model_dir : {model_dir}")
print(f"model.pt  : {model_pt}  ({model_pt.stat().st_size / 1024**2:.1f} MB)")

## 2b. Score-Threshold-Bestimmung (nur Live Run)

Im **Dry Run** wird der Standardwert `score_threshold = 0.7` verwendet.

Im **Live Run** wird der Threshold **datengetrieben** auf dem Dry-Run-Datensatz
bestimmt:
1. Dry-Run-Ref-DB (52 Songs) mit dem *neuen Live-Modell* indexieren
2. Dry-Run-Queries gegen diesen Index matchen (ohne Threshold)
3. F1-optimalen Punkt auf der Precision-Recall-Kurve bestimmen
4. Threshold für das Live-Run-Matching übernehmen

Die Dry-Run-Songs (52 Ref + 10 OOD) sind **disjunkt** zu den Live-Run-Songs
(live_ref ≈ 4930 + live_ood = 200) — kein Data Leakage.

In [ ]:
if RUN_MODE == "dry":
    # Dry run: use the default threshold from run_config
    score_threshold = cfg.score_threshold
    print(f"Dry run — using default score_threshold = {score_threshold}")

else:
    # Live run: determine optimal threshold on the dry-run validation set.
    # The dry-run songs are DISJOINT from the live-run songs → no data leakage.
    import numpy as np
    from sklearn.metrics import precision_recall_curve
    from utils import load_partitions, export_pfann_list
    from neural_fp import export_query_list, match_queries_timed

    DRY_PARTS_DIR   = PROJECT_ROOT / "data" / "partitions"
    DRY_MANIFEST    = PROJECT_ROOT / "data" / "queries" / "manifest_dry.csv"
    DRY_REF_LIST    = PFANN_LISTS / "dry_ref.txt"
    DRY_QUERY_LIST  = PFANN_LISTS / "dry_val_queries.txt"
    VAL_DB_DIR      = RESULTS_DIR / "pfann_val_db"
    VAL_TSV_PATH    = RESULTS_DIR / "neuralFP_val_matches.tsv"
    VAL_RAW_CSV     = RESULTS_DIR / "neuralFP_val_raw.csv"
    THRESHOLD_JSON  = RESULTS_DIR / "neuralFP_threshold.json"

    # ── Prerequisite: manifest_dry.csv must exist (created by NB 01 dry run) ──
    assert DRY_MANIFEST.exists(), (
        f"manifest_dry.csv fehlt: {DRY_MANIFEST}\n"
        "NB 01 im dry-run Modus ausführen!"
    )

    # ── Ensure dry_ref.txt exists (written by NB 04 dry run) ─────────────────
    if not DRY_REF_LIST.exists():
        print(f"dry_ref.txt not found, generating from dry_ref partition ...")
        _parts = load_partitions(DRY_PARTS_DIR)
        assert "dry_ref" in _parts, "dry_ref.json fehlt — NB 00 (dry) ausführen!"
        export_pfann_list(_parts["dry_ref"], PROJECT_ROOT / "data" / "fma_medium",
                          DRY_REF_LIST)
        print(f"  Geschrieben: {DRY_REF_LIST}")
    else:
        print(f"  dry_ref.txt vorhanden: {DRY_REF_LIST}")

    # ── Generate dry-run query list in manifest order ─────────────────────────
    dry_manifest = pd.read_csv(DRY_MANIFEST)
    export_query_list(dry_manifest, DRY_QUERY_LIST)
    print(f"  Dry-run query list: {len(dry_manifest)} queries → {DRY_QUERY_LIST}")

    # ── Build validation DB from dry_ref using the LIVE model ────────────────
    # Uses live_run.json (which points to checkpoints/live_run/model.pt)
    print("\nBuilding validation DB (dry_ref × live model) ...")
    VAL_DB_DIR.mkdir(parents=True, exist_ok=True)
    python_bin = sys.executable

    t_val_build = time.time()
    subprocess.run(
        [python_bin, "builder.py", str(DRY_REF_LIST), str(VAL_DB_DIR), str(CONFIG_PATH)],
        cwd=str(PFANN_DIR),
        check=True,
        capture_output=True,
        text=True,
    )
    print(f"  Validation DB built in {time.time() - t_val_build:.1f} s")

    # ── Match dry-run queries — score_threshold=0.0 keeps ALL pfann results ──
    print("\nMatching dry-run queries against validation DB ...")
    dry_raw_df = match_queries_timed(
        query_list_path = DRY_QUERY_LIST,
        db_dir          = VAL_DB_DIR,
        manifest        = dry_manifest,
        out_path        = VAL_RAW_CSV,
        score_threshold = 0.0,    # keep ALL scores; threshold applied below
        pfann_dir       = PFANN_DIR,
    )
    print(f"  Results: {len(dry_raw_df)} rows")

    # ── Compute optimal threshold via F1 on precision-recall curve ────────────
    labels      = []
    scores_list = []
    for _, row in dry_raw_df.iterrows():
        is_correct = (
            not bool(row["is_ood"]) and
            pd.notna(row["predicted_id"]) and
            int(row["predicted_id"]) == int(row["ref_track_id"])
        )
        labels.append(int(is_correct))
        scores_list.append(float(row["score"]) if pd.notna(row["score"]) else 0.0)

    labels      = np.array(labels)
    scores_arr  = np.array(scores_list)

    prec, rec, thresholds = precision_recall_curve(labels, scores_arr)
    f1 = 2 * prec * rec / (prec + rec + 1e-9)
    best_idx = int(f1.argmax())

    # precision_recall_curve returns len(thresholds) = len(prec) - 1
    # best_idx might point to the last element (no threshold); clip to valid range
    best_idx   = min(best_idx, len(thresholds) - 1)
    score_threshold = float(thresholds[best_idx])

    print(f"\nOptimaler Score-Threshold (F1-Maximum auf Dry-Run-Validierungsset):")
    print(f"  τ = {score_threshold:.4f}")
    print(f"  → F1   = {f1[best_idx]:.4f}")
    print(f"  → Prec = {prec[best_idx]:.4f}")
    print(f"  → Rec  = {rec[best_idx]:.4f}")
    print(f"  Validation set: {(labels == 1).sum()} positive / "
          f"{(labels == 0).sum()} negative samples")

    # ── Save threshold for reproducibility ────────────────────────────────────
    threshold_record = {
        "score_threshold": score_threshold,
        "method":          "F1-maximum on precision-recall curve",
        "validation_set":  "dry_run (52 ref songs, 10 OOD songs, disjoint from live_run)",
        "model":           str(CONFIG_PATH),
        "f1":              float(f1[best_idx]),
        "precision":       float(prec[best_idx]),
        "recall":          float(rec[best_idx]),
        "n_positive":      int((labels == 1).sum()),
        "n_negative":      int((labels == 0).sum()),
        "documentation":   (
            f"Der Score-Threshold wurde auf dem Dry-Run-Validierungsset "
            f"(52 Referenz-Songs, 10 OOD-Songs, disjunkt zum Live-Run) "
            f"als F1-optimaler Punkt bestimmt (τ = {score_threshold:.4f})."
        ),
    }
    with open(THRESHOLD_JSON, "w") as fh:
        json.dump(threshold_record, fh, indent=2)
    print(f"\nThreshold gespeichert: {THRESHOLD_JSON}")

    # ── Disjointness sanity check ──────────────────────────────────────────────
    _parts = load_partitions(DRY_PARTS_DIR)
    assert set(_parts.get("dry_ref", [])).isdisjoint(set(_parts.get("live_ref", []))), \
        "dry_ref und live_ref überschneiden sich — Data Leakage!"
    assert set(_parts.get("dry_ood", [])).isdisjoint(set(_parts.get("live_ood", []))), \
        "dry_ood und live_ood überschneiden sich — Data Leakage!"
    print("Disjointness dry_ref ∩ live_ref = ∅ ✓")
    print("Disjointness dry_ood ∩ live_ood = ∅ ✓")

print(f"\nVerwendeter score_threshold = {score_threshold:.4f}")

## 3. Manifest laden

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
print(f"Manifest: {len(manifest)} Queries")
print(f"  In-DB : {(~manifest['is_ood']).sum()}")
print(f"  OOD   : {manifest['is_ood'].sum()}")
print(f"  Gruppen: {sorted(manifest['group'].unique())}")
print()
print(manifest.head(3).to_string())

## 4. pfann-Datenbank aufbauen

`builder.py` liest die Referenz-Songs aus `dry_ref.txt`, berechnet Embeddings
mit dem trainierten Modell und schreibt die FAISS-Datenbank nach
`results/dry_run/pfann_db/`.

`build_time_s` wird für den Effizienzvergleich in NB 06 gespeichert.

In [ ]:
import sys
python_bin = sys.executable

build_cmd = [
    python_bin, "builder.py",
    str(REF_LIST),
    str(DB_DIR),
    str(CONFIG_PATH),
]
print("Kommando:", " ".join(build_cmd))
print(f"cwd: {PFANN_DIR}")

t0 = time.time()
build_result = subprocess.run(
    build_cmd,
    cwd=str(PFANN_DIR),
    capture_output=True,
    text=True,
    check=True,
)
build_time_s = time.time() - t0

print(f"\nDB aufgebaut in {build_time_s:.1f} s  ({build_time_s / 60:.2f} Min.)")
print("--- stdout (letzte 20 Zeilen) ---")
for line in build_result.stdout.strip().splitlines()[-20:]:
    print(line)

db_files = list(DB_DIR.iterdir())
print(f"\nDB-Verzeichnis: {DB_DIR}")
for f in sorted(db_files):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")

In [ ]:
print(f"\nDB aufgebaut in {build_time_s:.1f} s  ({build_time_s / 60:.2f} Min.)")

In [ ]:
print(f"\nDB aufgebaut in {build_time_s:.1f} s  ({build_time_s / 60:.2f} Min.)")
print("--- stdout (letzte 20 Zeilen) ---")
for line in build_result.stdout.strip().splitlines()[-20:]:
    print(line)

db_files = list(DB_DIR.iterdir())
print(f"\nDB-Verzeichnis: {DB_DIR}")
for f in sorted(db_files):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")

## 5. Query-Liste erzeugen

`export_query_list()` schreibt absolute WAV-Pfade in Manifest-Reihenfolge
nach `pfann/lists/dry_queries.txt`. Die Reihenfolge ist kritisch:
TSV- und detail.csv-Zeilen werden später positionsbasiert zugeordnet.

In [ ]:
from neural_fp import export_query_list

export_query_list(manifest, QUERY_LIST)

with open(QUERY_LIST) as fh:
    sample_lines = fh.readlines()
print(f"Erste 3 Zeilen von {QUERY_LIST.name}:")
for line in sample_lines[:3]:
    print(f"  {line.rstrip()}")
print(f"  ... ({len(sample_lines)} Einträge gesamt)")

## 6. Queries matchen (per-Query FAISS-Timing)

`match_queries_timed()` repliziert die pfann-Matcher-Logik, misst aber
die FAISS-Lookup-Zeit (search + reranking) pro Query einzeln via
`time.perf_counter()`. Embedding-Generierung wird NICHT mitgemessen —
gleiche Methodik wie Shazam/Quad (nur DB-Lookup getimed).

In [ ]:
from neural_fp import match_queries_timed

score_threshold = 0.54

raw_df = match_queries_timed(
    query_list_path = QUERY_LIST,
    db_dir          = DB_DIR,
    manifest        = manifest,
    out_path        = RAW_CSV_PATH,
    score_threshold = score_threshold,   # datengetrieben (live) oder 0.7 (dry)
    pfann_dir       = PFANN_DIR,
)

print(f"score_threshold used: {score_threshold:.4f}")
print(f"\nShape: {raw_df.shape}")
print(raw_df.head(5).to_string())
times = raw_df["query_time_ms"].dropna()
    print(f"\nneuralFP raw result CSV: {out_path} ({len(output_df)} rows)")
    if len(times) > 0:
        print(f"  query_time_ms — mean: {times.mean():.2f}, "
              f"median: {times.median():.2f}, std: {times.std():.2f}, "
              f"p95: {times.quantile(0.95):.2f}")

## 8. Sanity Checks

In [ ]:
print("=== Sanity Checks ===")

# 1. Anzahl Zeilen = Anzahl Manifest-Einträge
assert len(raw_df) == len(manifest), (
    f"Zeilenanzahl: raw_df={len(raw_df)}, manifest={len(manifest)}"
)
print(f"  [OK] Zeilenzahl: {len(raw_df)} == manifest-Einträge")

# 2. result_class-Verteilung
class_counts = raw_df["result_class"].value_counts()
print(f"\n  result_class Verteilung:")
for cls, cnt in class_counts.items():
    print(f"    {cls}: {cnt}")

# 3. Keine pred_id=0
zero_mask = raw_df["predicted_id"] == 0
assert not zero_mask.any(), (
    f"pred_id=0 gefunden in {zero_mask.sum()} Zeilen — "
    "leere Matches wurden nicht korrekt auf None gemappt!"
)
print(f"\n  [OK] Keine pred_id=0 (leere Matches korrekt als None)")

# 4. OOD-Queries haben ref_track_id = None
ood_mask = raw_df["is_ood"]
ood_ref_nulls = raw_df.loc[ood_mask, "ref_track_id"].isna()
assert ood_ref_nulls.all(), (
    f"{(~ood_ref_nulls).sum()} OOD-Queries haben ref_track_id != None!"
)
print(f"  [OK] Alle {ood_mask.sum()} OOD-Queries haben ref_track_id=None")

# 5. Threshold-Info
print(f"\n  Score-Threshold: {score_threshold:.4f}", end="")
if RUN_MODE == "live":
    thresh_file = RESULTS_DIR / "neuralFP_threshold.json"
    print(f"  (datengetrieben, gespeichert in {thresh_file.name})")
else:
    print(f"  (Standardwert aus run_config)")

# 6. Hit Rate / Specificity Vorschau
in_db  = raw_df[~raw_df["is_ood"]]
ood_df = raw_df[raw_df["is_ood"]]
if len(in_db) > 0:
    hit_rate = (in_db["result_class"] == "TP").sum() / len(in_db)
    print(f"\n  Hit Rate     : {hit_rate:.3f}  (In-DB queries)")
if len(ood_df) > 0:
    specificity = (ood_df["result_class"] == "TN").sum() / len(ood_df)
    print(f"  Specificity  : {specificity:.3f}  (OOD queries)")
print(f"  Output CSV   : {RAW_CSV_PATH}")